# Comfort_indicators

Hours each zone is outside the setpoints of its system, with the gallery example 06 (DX heat pump in a box in Sevilla, 20-25 °C while the office is open).

In [1]:
import json, os
import opensimula as osm
from opensimula import gui

os.makedirs("comfort_example", exist_ok=True)
gui.copy_examples("comfort_example")
os.chdir("comfort_example")

Examples in /Users/jfc/Documents/opensimula/jupyter_test/comfort_example:
  01_schedules.json
  02_weather.json
  03_walls.json
  04_windows_and_gains.json
  05_ideal_system.json
  06_dx_system.json
  07_fan_coil.json
  08_load_records.json
  09_water_plant.json
  office_loads.csv
  sevilla.met

Open one with:
  cd comfort_example
  opensimula gui 01_schedules.json


Example 06 plus `Comfort_indicators` (tolerance 0.5 K, hours counted while the system is on) and `Energy_indicators`.

In [2]:
project = json.load(open("06_dx_system.json", encoding="utf-8"))
project["components"] += [
    {"type": "Comfort_indicators", "name": "comfort", "tolerance": 0.5},
    {"type": "Energy_indicators", "name": "energy"},
]
sim = osm.Simulation()
pro = sim.new_project("06")
pro.read_dict(project)
pro.simulate()

Reading project data from dictonary
Reading completed.
Checking project: 06 DX system
Checking completed.
Calculating solar direct shadows ...
Simulating 06 DX system: ...


100%|██████████| 8760/8760 [00:14<00:00, 612.33step/s, n_iter=2] 


In [3]:
pro.component("comfort").report()

Comfort indicators: comfort
Hours outside the setpoints: 18 of 2,380 zone-hours counted (0.8%)
Tolerance: 0.5 K; hours counted: system on

Comfort by zone
        Hours counted  Hours above  Hours below  Hours outside  Outside [%]  Degree-hours above [K·h]  Degree-hours below [K·h]  T max [°C]  T min [°C]
zone                                                                                                                                                   
office        2,380.0         18.0          0.0           18.0          0.8                      23.2                       0.0        28.1        20.0
TOTAL         2,380.0         18.0          0.0           18.0          0.8                      23.2                       0.0        28.1        20.0

Hours outside the setpoints by month and zone [h]
          office  TOTAL
month                  
Jan 2001     0.0    0.0
Feb 2001     0.0    0.0
Mar 2001     0.0    0.0
Apr 2001     0.0    0.0
May 2001     0.0    0.0
Jun 2001     0.0    0.0
Jul 2001     0.0    0.0
Aug 2001     0.0    0.0
Sep 2001    11.0   11.0
Oct 2001     7.0    7.0
Nov 2001     0.0    0.0
Dec 2001     0.0    0.0
Total       18.0   18.0

Hours of each zone counted while its system is on (and the period is not 0); the TOTAL row adds the zone-hours of every zone.

Only the occupied hours (`period`, an expression with `input_variables`) and a tolerance of 0.1 K (the zone is held at its setpoint within the convergence of the simulation, 0.01 K: with no tolerance those hours would count):

In [4]:
occupied = json.load(open("06_dx_system.json", encoding="utf-8"))
occupied["components"].append({
    "type": "Comfort_indicators", "name": "comfort", "tolerance": 0.1,
    "input_variables": ["f = office_hours.values"], "period": "f",
})
pro_occ = sim.new_project("06 occupied")
pro_occ.read_dict(occupied)
pro_occ.simulate()
pro_occ.component("comfort").report()

Reading project data from dictonary
Reading completed.
Checking project: 06 DX system
Checking completed.
Calculating solar direct shadows ...
Simulating 06 DX system: ...


100%|██████████| 8760/8760 [00:14<00:00, 614.69step/s, n_iter=2] 


Comfort indicators: comfort
Hours outside the setpoints: 25 of 2,380 zone-hours counted (1.1%)
Tolerance: 0.1 K; hours counted: system on and period = f

Comfort by zone
        Hours counted  Hours above  Hours below  Hours outside  Outside [%]  Degree-hours above [K·h]  Degree-hours below [K·h]  T max [°C]  T min [°C]
zone                                                                                                                                                   
office        2,380.0         25.0          0.0           25.0          1.1                      25.6                       0.0        28.1        20.0
TOTAL         2,380.0         25.0          0.0           25.0          1.1                      25.6                       0.0        28.1        20.0

Hours outside the setpoints by month and zone [h]
          office  TOTAL
month                  
Jan 2001     0.0    0.0
Feb 2001     0.0    0.0
Mar 2001     0.0    0.0
Apr 2001     0.0    0.0
May 2001     0.0    0.0
Jun 2001     0.0    0.0
Jul 2001     0.0    0.0
Aug 2001     0.0    0.0
Sep 2001    14.0   14.0
Oct 2001    11.0   11.0
Nov 2001     0.0    0.0
Dec 2001     0.0    0.0
Total       25.0   25.0

Hours of each zone counted while its system is on (and the period is not 0); the TOTAL row adds the zone-hours of every zone.

Hourly variables: `outside_office` (1 above, -1 below the setpoints, 0 inside or not counted) and `zones_outside`.

In [5]:
pro.results()

In [6]:
pro.component("energy").report()

,Final [kWh],Primary ren [kWh],Primary nren [kWh],Primary total [kWh],CO2 [kg]
service,,,,,
HEATING,142.6,209.8,278.7,488.5,47.2
COOLING,"1,855.2",768.1,"3,625.1","4,393.1",614.1
AIR_TRANSPORT,571.2,236.5,"1,116.1","1,352.6",189.1
LIGHTING,"1,142.4",473.0,"2,232.2","2,705.2",378.1
EQUIPMENT,913.9,378.4,"1,785.8","2,164.2",302.5
TOTAL,"4,625.4","2,065.6","9,037.9","11,103.5","1,531.0"
,Final [kWh/m²],Primary ren [kWh/m²],Primary nren [kWh/m²],Primary total [kWh/m²],CO2 [kg/m²]
service,,,,,
HEATING,3.0,4.4,5.8,10.2,1.0
